# 📒 Project 4 - Contact Book

## What it demonstrates
| Concept | Where used |
|---------|------------|
| Dataclasses | `Contact` model with validation |
| File I/O + JSON | Save and load contacts from disk |
| CRUD operations | Create, Read, Update, Delete |
| Exception handling | Duplicate contacts, missing keys |
| Regex | Phone and email format validation |
| Sorting & searching | Filter contacts by name/tag/phone |
| `@property` | Full name computed from parts |

## CRUD operations
```
Create  → add_contact()
Read    → find_contact(), list_contacts(), search()
Update  → update_contact()
Delete  → delete_contact()
```

## Storage format (JSON)
```json
{
  "Purvi Jain": {
    "first_name": "Purvi",
    "last_name":  "Jain",
    "phone":      "9876543210",
    "email":      "purvi@example.com",
    "tags":       ["friend", "college"]
  }
}
```

In [1]:
# ============================================================
#  PROJECT 4 — CONTACT BOOK
# ============================================================

import json, re, os
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict

PHONE_RE = re.compile(r"^[6-9]\d{9}$")                    # Indian mobile
EMAIL_RE = re.compile(r"^[\w.+-]+@[\w-]+\.[\w.]+$", re.I) # basic email

@dataclass
class Contact:
    first_name: str
    last_name:  str
    phone:      str
    email:      str  = ""
    tags:       List[str] = field(default_factory=list)

    def __post_init__(self):
        self.first_name = self.first_name.strip().title()
        self.last_name  = self.last_name.strip().title()
        phone_clean     = re.sub(r"[\s\-()]", "", self.phone)
        if not PHONE_RE.match(phone_clean):
            raise ValueError(f"Invalid phone number: {self.phone!r}")
        self.phone = phone_clean
        if self.email and not EMAIL_RE.match(self.email):
            raise ValueError(f"Invalid email: {self.email!r}")
        self.tags = [t.lower().strip() for t in self.tags]

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}".strip()

    def display(self) -> str:
        tag_str = ", ".join(self.tags) if self.tags else "—"
        return (
            f"  ┌─ {self.full_name}\n"
            f"  │  📞 {self.phone}\n"
            f"  │  ✉️  {self.email or '—'}\n"
            f"  └─ 🏷  {tag_str}"
        )

class ContactBook:
    """CRUD contact manager with JSON persistence."""

    def __init__(self, filepath: str = "/tmp/contacts.json"):
        self.filepath  = filepath
        self._contacts: Dict[str, Contact] = {}
        self._load()

    # ---- Persistence ----
    def _load(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath) as f:
                    raw = json.load(f)
                for key, data in raw.items():
                    self._contacts[key] = Contact(**data)
            except (json.JSONDecodeError, TypeError):
                self._contacts = {}

    def save(self):
        with open(self.filepath, "w") as f:
            json.dump(
                {k: asdict(v) for k, v in self._contacts.items()},
                f, indent=2
            )
        print(f"  💾 Saved {len(self._contacts)} contacts to {self.filepath!r}")

    # ---- CRUD ----
    def add(self, contact: Contact):
        key = contact.full_name
        if key in self._contacts:
            raise ValueError(f"Contact {key!r} already exists. Use update() to modify.")
        self._contacts[key] = contact
        print(f"  ✅ Added: {key}")

    def get(self, name: str) -> Optional[Contact]:
        return self._contacts.get(name.title())

    def update(self, name: str, **fields):
        key = name.title()
        if key not in self._contacts:
            raise KeyError(f"Contact {key!r} not found.")
        old = asdict(self._contacts[key])
        old.update(fields)
        self._contacts[key] = Contact(**old)
        print(f"  ✏️  Updated: {key}")

    def delete(self, name: str):
        key = name.title()
        if key not in self._contacts:
            raise KeyError(f"Contact {key!r} not found.")
        del self._contacts[key]
        print(f"  🗑️  Deleted: {key}")

    # ---- Search & list ----
    def search(self, query: str) -> List[Contact]:
        q = query.lower()
        return [
            c for c in self._contacts.values()
            if q in c.full_name.lower()
            or q in c.phone
            or q in c.email.lower()
            or any(q in t for t in c.tags)
        ]

    def list_all(self, sort_by: str = "name"):
        contacts = list(self._contacts.values())
        if sort_by == "name":
            contacts.sort(key=lambda c: c.full_name)
        elif sort_by == "phone":
            contacts.sort(key=lambda c: c.phone)
        return contacts

    def filter_by_tag(self, tag: str) -> List[Contact]:
        return [c for c in self._contacts.values() if tag.lower() in c.tags]

    def __len__(self):  return len(self._contacts)
    def __repr__(self): return f"ContactBook({len(self)} contacts)"

print("✅ ContactBook defined.")

✅ ContactBook defined.


In [2]:
# ---- Demo — full CRUD session ----

print("=" * 52)
print("         📒  CONTACT BOOK DEMO")
print("=" * 52)

book = ContactBook("/tmp/demo_contacts.json")

# CREATE
print("\n➕ Adding contacts...")
contacts_data = [
    Contact("Purvi",   "Jain",  "9876543210", "purvi@example.com",  ["friend", "college"]),
    Contact("Alice",   "Sharma","9123456789", "alice@gmail.com",    ["work", "college"]),
    Contact("Bob",     "Kumar", "8765432190", "bob@outlook.com",    ["family"]),
    Contact("Charlie", "Singh", "7654321098", "",                   ["work"]),
    Contact("Diana",   "Patel", "6543210987", "diana@example.in",   ["friend"]),
]
for c in contacts_data:
    book.add(c)

# Duplicate test
try:
    book.add(Contact("Purvi", "Jain", "9876543210"))
except ValueError as e:
    print(f"  ⚠️  {e}")

# Invalid phone
try:
    Contact("Bad", "Phone", "1234567890")
except ValueError as e:
    print(f"  ⚠️  {e}")

# READ — list all
print(f"\n📋 All contacts ({len(book)} total, sorted by name):")
for c in book.list_all():
    print(c.display())
    print()

# SEARCH
print("🔍 Search 'college':")
for c in book.search("college"):
    print(f"  → {c.full_name} ({', '.join(c.tags)})")

print("\n🔍 Search 'work' tag:")
for c in book.filter_by_tag("work"):
    print(f"  → {c.full_name}")

# UPDATE
print("\n✏️  Updating Charlie Singh — adding email and family tag...")
book.update("Charlie Singh", email="charlie@example.com", tags=["work", "friend"])
print(book.get("Charlie Singh").display())

# DELETE
print("\n🗑️  Deleting Diana Patel...")
book.delete("Diana Patel")
print(f"  Total after delete: {len(book)}")

# SAVE
print()
book.save()

# RELOAD from file
book2 = ContactBook("/tmp/demo_contacts.json")
print(f"  Reloaded from disk: {book2}")

print("\n" + "=" * 52)

         📒  CONTACT BOOK DEMO

➕ Adding contacts...
  ✅ Added: Purvi Jain
  ✅ Added: Alice Sharma
  ✅ Added: Bob Kumar
  ✅ Added: Charlie Singh
  ✅ Added: Diana Patel
  ⚠️  Contact 'Purvi Jain' already exists. Use update() to modify.
  ⚠️  Invalid phone number: '1234567890'

📋 All contacts (5 total, sorted by name):
  ┌─ Alice Sharma
  │  📞 9123456789
  │  ✉️  alice@gmail.com
  └─ 🏷  work, college

  ┌─ Bob Kumar
  │  📞 8765432190
  │  ✉️  bob@outlook.com
  └─ 🏷  family

  ┌─ Charlie Singh
  │  📞 7654321098
  │  ✉️  —
  └─ 🏷  work

  ┌─ Diana Patel
  │  📞 6543210987
  │  ✉️  diana@example.in
  └─ 🏷  friend

  ┌─ Purvi Jain
  │  📞 9876543210
  │  ✉️  purvi@example.com
  └─ 🏷  friend, college

🔍 Search 'college':
  → Purvi Jain (friend, college)
  → Alice Sharma (work, college)

🔍 Search 'work' tag:
  → Alice Sharma
  → Charlie Singh

✏️  Updating Charlie Singh — adding email and family tag...
  ✏️  Updated: Charlie Singh
  ┌─ Charlie Singh
  │  📞 7654321098
  │  ✉️  charlie@example.com


FileNotFoundError: [Errno 2] No such file or directory: '/tmp/demo_contacts.json'